# B5T1 part B in a notebook: eval a prompt, let GEPA rewrite it, eval it again

This notebook follows [README.md](README.md) one cell at a time. The README is the full guide and gives the same
steps as terminal commands. In each step below, run the cells in place of those commands. Start at the top and
run the cells in order.

Before each step, write your prediction in your notes. After you run the step's cells, open its
**Check your predictions** box and compare it with what you saw.

The cells call the kit's files: [`check_gateway.py`](check_gateway.py), [`task.py`](task.py) (with Inspect), [`optimize.py`](optimize.py), [`budget.py`](budget.py) and
[`compare.py`](compare.py). In your editor, open one of those linked files to see what its cell does.

## Set up

In the notebook, select this folder's `.venv` kernel. In VS Code, choose **Select Kernel** ›
**Python Environments** › `.venv`.

In the notebook, run the next cell. If you started Jupyter from a terminal where `KB_MCP_TOKEN` is set, the cell does not ask for a token.
Otherwise, enter your token in the hidden box the cell opens. Do not type the token into a cell's code. A saved
notebook keeps its cells.

In [ ]:
import getpass
import os

if not os.environ.get("KB_MCP_TOKEN"):
    os.environ["KB_MCP_TOKEN"] = getpass.getpass("Paste your course token, then press Enter: ")
print("KB_MCP_TOKEN is set" if os.environ.get("KB_MCP_TOKEN") else "KB_MCP_TOKEN is NOT set")

In the output of the previous notebook cell, you should see `KB_MCP_TOKEN is set`.

In the notebook, run the next cell.

In [ ]:
import check_gateway

problem = check_gateway.main()

In the output of the previous notebook cell, you should see ✓ lines and `OK: the gateway works with your token.` A line starting `PROBLEM:` tells you what to fix.

## Step 1: run the starting prompt

<details>
<summary>What happens when you do this step</summary>

```text
prompt.txt + held-out test cases -> task model -> SQL -> code scorer
```

Inspect runs the task model once per case with [`prompt.txt`](prompt.txt) as its system prompt. The code scorer runs the
SQL and compares its number with the reference.

</details>

In the notebook, run the next cell.

In [ ]:
print(open("prompt.txt", encoding="utf-8").read())

In the output of the previous notebook cell, you should see the starting prompt from [`prompt.txt`](prompt.txt): the table and its columns, and the instruction to write one query. This is the system prompt the task model gets.

In the notebook, run the next cell.

In [ ]:
import json

with open("data/cases.jsonl", encoding="utf-8") as f:
    cases = [json.loads(line) for line in f]

test_questions = [c["question"] for c in cases if c["split"] == "test"]
for q in test_questions[:4]:
    print("-", q)

In the output of the previous notebook cell, you should see four of the held-out questions from [`data/cases.jsonl`](data/cases.jsonl). These are the cases the eval runs.

**Predict:** Now that you have read the prompt and some of the questions, write down in your notes whether you expect the starting prompt to pass below half, about half, or more than half
of the held-out cases. What kind of question do you think might fail, given what the prompt says and does not say about the table?

In the notebook, run the next cell.

In [ ]:
import inspect_ai

from task import sql_cases

before_logs = inspect_ai.eval(
    sql_cases(split="test", prompt="prompt.txt"),
    log_dir="logs",
    display="none",
)
before = before_logs[0]
print("eval finished:", before.status, "-", len(before.samples), "held-out cases")

In the output of the previous notebook cell, you should see `eval finished: success - 12 held-out cases`. The cell does not print the score yet.

<details>
<summary>Check your predictions</summary>

The score is the fraction of held-out cases that pass. Each case contributes the same share. In L15, an Inspect
eval has a dataset, a solver and a scorer: here the cases supply inputs and targets, the task model supplies SQL, and
code checks the result.
</details>

## Step 2: read and name the failures

<details>
<summary>What happens when you do this step</summary>

```text
held-out run -> failed cases -> failure names -> score
```

You read each failed case and name what went wrong before you look at the score, so the aggregate does not
steer what you look for. Treat the reference SQL and value as correct for this data.

</details>

A failure name is short and says what to fix. You give a failed case a name, and you reuse that name when another
case fails in the same way. The next cell prints the failed cases; a later cell prints the score.

**Predict:** In your notes, write down whether you expect several failed cases to share a failure name or each
failed case to need its own name.

In the notebook, run the next cell.

In [ ]:
for sample in before.samples:
    check = sample.scores["sql_check"]
    if check.value == 0:
        print("case", sample.id)
        print(check.explanation)
        print("-" * 70)

In the output of the previous notebook cell, you should see a block for each failed case. Each block gives the question, the reference SQL and value, and `FAILURE:` with what the checker found and the model's SQL.

**Your error-analysis table.** In the notebook, double-click this cell to edit it. Write one row for each failed case in your own words.

| case id | what went wrong | name of the failure |
|---|---|---|
| | | |
| | | |
| | | |

In the notebook, run the next cell.

In [ ]:
for score in before.results.scores:
    if score.name == "sql_check":
        print("held-out score, starting prompt:", round(score.metrics["accuracy"].value, 3))

In the output of the previous notebook cell, you should see the score: the share of held-out cases that passed, between 0 and 1. In the notebook's error-analysis table, count how many cases each failure name covers.

<details>
<summary>Check your predictions</summary>

Cases with the same failure should have the same name, and the counts across your names should add up to the
number of failed cases. Reading the failures before the score is error analysis: the cases tell you what to change,
and the score only tells you how many cases need it.
</details>

## Step 3: let GEPA rewrite the prompt

<details>
<summary>What happens when you do this step</summary>

```text
train cases -> failures -> reflection model -> prompt candidates -> val cases -> selected prompt
```

GEPA uses train cases and their feedback to propose and test prompt changes. It keeps a Pareto set of candidates
that each do best on some examples and returns a prompt selected using val scores; GEPA does not read the test
cases.

</details>

**Predict:** In your notes, write down whether you expect the new prompt to address none, some or all of your
failure names. Do you expect the new prompt to state, in its own words, a rule you found in Step 2?

In the notebook, run the next cell.

In [ ]:
import optimize

stopped = optimize.main([])

In the output of the previous notebook cell, you should see the estimate first, followed by lines like `Iteration 1: Proposed new text for predict:` for each new prompt. At the end of that output, you should see the val scores, the cost in CAD and `wrote prompt.optimized.txt`. If the cost meter stops GEPA, the cell says so and still writes the best prompt GEPA had. In your editor's `evals-gepa` folder file list, you should see `prompt.optimized.txt` next to [`prompt.txt`](prompt.txt).

<details>
<summary>Check your predictions</summary>

GEPA uses train and val cases. It does not read any held-out test case. L17's evaluator-optimizer
pattern uses feedback to propose revisions and a limit to stop attempts. A prompt change can address a failure
without guaranteeing a better held-out score.
</details>

## Step 4: read the new prompt

<details>
<summary>What happens when you do this step</summary>

```text
prompt.txt + prompt.optimized.txt -> added lines -> failure names and data check
```

A prompt can improve an eval and still contain a rule that is not true of the data. Read the added lines before
using the prompt and check their claims.

</details>

**Predict:** In your notes, write down whether you expect every failure name in your table to be addressed, and whether you
expect every added rule to be true of the data.

In the notebook, run the next cell.

In [ ]:
import difflib

old = open("prompt.txt", encoding="utf-8").read().splitlines()
new = open("prompt.optimized.txt", encoding="utf-8").read().splitlines()
for line in difflib.unified_diff(old, new, "prompt.txt", "prompt.optimized.txt", lineterm=""):
    print(line)

In the output of the previous notebook cell, you should see file headers starting `---` and `+++`. Below them, lines starting `-` show what GEPA removed and lines starting `+` show what GEPA added.

1. In the output of the previous notebook cell, read the `+` lines and match each added rule to a failure name from your notebook table in Step 2. You should see which failure names the new prompt addresses.
2. In the notebook, change the query in the next cell to test one added rule against the data. Run that cell. You should see the query result in its output. In your notes, mark any rule you cannot confirm: an optimiser writes rules that fit the cases, and you check them before you rely on them.

In [ ]:
import duckdb

# Change this query to test one rule from the new prompt against the data.
duckdb.sql("""
    SELECT *
    FROM 'data/trips.csv'
    LIMIT 5
""").show()

<details>
<summary>Check your predictions</summary>

An added line can address a failure name, leave a name unaddressed, or make a claim you cannot verify from the
data script. L17's evaluator-optimizer pattern checks a result against criteria. A reader still needs to check
claims before using the prompt.
</details>

## Step 5: eval the new prompt

<details>
<summary>What happens when you do this step</summary>

```text
prompt.optimized.txt + same held-out test cases -> task model -> code scorer
```

Run the same eval on the same test split so the scores and individual cases can be compared. GEPA selected a
prompt using val cases, so the held-out cases provide a separate check.

</details>

**Predict:** In your notes, write down whether you expect the held-out score to be higher, lower, or about the same, and whether
you expect the same or fewer failures.

In the notebook, run the next cell.

In [ ]:
after_logs = inspect_ai.eval(sql_cases(split="test", prompt="prompt.optimized.txt"), log_dir="logs",
                            display="none")
after = after_logs[0]
print("eval finished:", after.status, "-", len(after.samples), "held-out cases")
still_failing = 0
for sample in after.samples:
    check = sample.scores["sql_check"]
    if check.value == 0:
        print("case", sample.id)
        print(check.explanation)
        print("-" * 70)
        still_failing = still_failing + 1
print("cases that still fail:", still_failing)
for score in after.results.scores:
    if score.name == "sql_check":
        print("held-out score, new prompt:", round(score.metrics["accuracy"].value, 3))

In the output of the previous notebook cell, you should see `eval finished: success - 12 held-out cases` and a block for each case that still fails, if any. The last two lines give the number of cases that still fail and the new prompt's score on the held-out cases.

Compare the failures in that output with the names in your Step 2 notebook table. Does each remaining failure have a name in your table? If none fail, which names did the new prompt fix? Compare the new score in that output with the starting score in the Step 2 score cell's output.

<details>
<summary>Check your predictions</summary>

Compare whether the after score is at least the before score and whether the new prompt has the same or fewer
failures. Either relation can fail on a particular run. L17 uses held-out cases because GEPA chose the prompt on
validation cases, so the held-out score checks questions outside that choice.
</details>

## Step 6: price the session

<details>
<summary>What happens when you do this step</summary>

```text
upper-bound estimate -> eval and GEPA records -> actual cost comparison
```

The estimate assumes every reply uses its full token cap, so it is an upper bound. `compare.py` adds up the
tokens the gateway reported for each call and prices them, so the recorded cost is at or below the estimate.

</details>

Your token's term budget is 2 CAD. The course rule for one whole run (check, eval, GEPA, eval again) is less than
0.10 CAD. [`budget.py`](budget.py) prints an upper-bound estimate of the calls and their cost before any call. [`optimize.py`](optimize.py)
printed an estimate at the start of Step 3. [`compare.py`](compare.py) reads `logs/` and reports the actual cost.

In the notebook, run the next cell.

In [ ]:
import budget

estimate = budget.print_estimate()

In the output of the previous notebook cell, you should see the estimate: the calls and the most each part of the run can cost, before any call.

**Predict:** In your notes, write down how far below this estimate you expect the recorded total to be: about the
same, around half, or a small fraction. Do you expect the whole session to cost a few cents or less?

In the notebook, run the next cell.

In [ ]:
import compare

missing = compare.main()

In the output of the previous notebook cell, you should see a line for each eval run with its score and cost in CAD, the cost of GEPA, the total, and the check line on the held-out cases.

Compare the recorded total in that output with the estimate in the `budget.py` notebook cell's output.

<details>
<summary>Check your predictions</summary>

The recorded total should be below the upper-bound estimate for the corresponding run. L17's course LLM gateway
reserves the most a call can cost before the call and replaces the reserve with the actual cost after the reply;
bounded attempts help keep spending within a limit.
</details>

## Next

Go on with **4. Extra exercises** and **6. Recap and reflection** in [README.md](README.md).